# Time-Series Analyses: ITS, RDiT and GEE
## Is there a step change when students return to school?

These exploratory approaches examine the daily series averaged across monitoring zones:

- ITS: trend, Fourier seasonality and ARMA errors; does school-term status shift the level?
- Cutoff-specific jumps: segmented regression within ±60 days of each return date, using HAC errors.
- RDiT: windows of ±28/±42/±56 days with a ±2-day donut around each return date, plus reference cutoffs at vacation midpoints. Dates come from the official calendar in notebook 01.
- GEE with within-unit AR(1) correlation: a supplementary approach with only six clusters, requiring caution.

**Data access.** The SIMA measurements used here are not
redistributed in this repository (SIMA provided them for this study on the condition
that they are not disseminated); the repository holds the code, the official school
calendar and all derived results. This notebook reads files that notebooks 00 and 01
build from those measurements, and its configuration cell stops with a clear message
if they are missing. To rebuild them, request the annual workbooks from SIMA, place
them in `data_raw/` and run notebook 00 (its header gives the exact file format).

**Role in the analysis.** Secondary exploratory analyses (ITS, RDiT, GEE): kept for completeness and reproducibility. They are not reported in the article and do not define its conclusions. The analytical hierarchy of the whole repository is stated in the README.

## 1. Initial setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.genmod.cov_struct import Autoregressive
from statsmodels.genmod.generalized_estimating_equations import GEE
import json
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

contaminantes = ['CO', 'NO2', 'O3', 'PM10', 'PM2.5']
colores_contaminantes = {'CO': '#3498db', 'NO2': '#e74c3c', 'O3': '#2ecc71',
                         'PM10': '#f39c12', 'PM2.5': '#9b59b6'}

output_dir = Path('../data_processed')
resultados_dir = output_dir / 'resultados_metodos'
graficos_dir = output_dir / 'graficos_metodos'

print('Libraries imported successfully')

# Check required inputs (restricted data; see heading)
# Input-data check (restricted data; see heading)
archivos_requeridos = [output_dir / 'panel_pico_hibrido.csv',
                       output_dir / 'inicios_bloques_clase.csv',
                       output_dir / 'cortes_placebo.csv']
faltantes = [str(a) for a in archivos_requeridos if not a.exists()]
if faltantes:
    raise FileNotFoundError(
        ''
        ''
        "Files derived from the SIMA measurements are missing; they are not redistributed in this repository. "
        "Request the annual workbooks from SIMA, place them in data_raw/ and run notebooks 00 and 01. "
        f"Missing: {faltantes}")
print(f"Input data found: {len(archivos_requeridos)} files")

Libraries imported successfully
Input data found: 3 files


## 2. Daily series and calendar cutoffs

In [2]:
panel = pd.read_csv('../data_processed/panel_pico_hibrido.csv', parse_dates=['date'])
columnas_log = [f'l_{c}' for c in contaminantes]
columnas_fourier = [f'{f}{k}' for f in ('sin', 'cos') for k in (1, 2, 3)]

serie_diaria = (panel.groupby('date')
                .agg({**{col: 'mean' for col in columnas_log}, 'PC1': 'mean',
                      'clase': 'first', 'dow': 'first', 't': 'first', 'mes': 'first',
                      **{col: 'first' for col in columnas_fourier}})
                .reset_index().sort_values('date').reset_index(drop=True))

inicios = pd.read_csv('../data_processed/inicios_bloques_clase.csv', parse_dates=['inicio_clases'])
cortes_clase = [str(d.date()) for d in inicios['inicio_clases']]
placebos = pd.read_csv('../data_processed/cortes_placebo.csv', parse_dates=['corte_placebo'])
cortes_placebo = [str(d.date()) for d in placebos['corte_placebo']]

print("=" * 70)
print('DAILY SERIES AND CUTOFFS')
print("=" * 70)
print(f"Daily series: {len(serie_diaria)} days")
print(f"Term starts (official calendar): {cortes_clase}")
print(f"Reference cutoffs (vacation midpoints): {cortes_placebo}")

DAILY SERIES AND CUTOFFS
Daily series: 1096 days
Term starts (official calendar): ['2023-01-09', '2023-08-28', '2024-01-08', '2024-08-26', '2025-01-09', '2025-09-01']
Reference cutoffs (vacation midpoints): ['2023-08-11', '2023-12-27', '2024-03-30', '2024-08-05', '2024-12-29', '2025-08-08']


## 3. ITS: school-term status with ARMA errors selected by AIC

### ITS equation

For the daily series averaged over the six zones, include a trend, Fourier seasonality and school-term status:

$$ \log y_t=\alpha+\beta\text{School}_t+\lambda t+\sum_{k=1}^K\left[a_k\sin\frac{2\pi kt}{365.25}+b_k\cos\frac{2\pi kt}{365.25}\right]+\delta_{d(t)}+u_t,\qquad u_t\sim ARMA(p,q). $$

Sines and cosines represent smooth annual seasonality. Error orders $(p,q)$ are selected by AIC separately for each pollutant. Serial dependence is modeled explicitly rather than assuming independent daily residuals; inference on $\beta$ remains conditional on those modeling choices.

In [3]:
X_its = pd.concat([serie_diaria[['clase']].astype(float),
                   (serie_diaria['t'] / 365.25).rename('tendencia'),
                   pd.get_dummies(serie_diaria['dow'], prefix='dow', drop_first=True).astype(float),
                   serie_diaria[columnas_fourier]], axis=1)
X_its = sm.add_constant(X_its)
ordenes_arma = [(1, 0, 0), (2, 0, 0), (1, 0, 1), (2, 0, 1)]

resultados_its = {}
print("=" * 70)
print('ITS: SCHOOL-TERM INDICATOR (trend + Fourier + weekday + ARMA errors selected by AIC)')
print("=" * 70)
for c in contaminantes + ['PC1']:
    y = serie_diaria[f'l_{c}'] if c != 'PC1' else serie_diaria['PC1']
    mejor = None
    for orden in ordenes_arma:
        try:
            modelo = SARIMAX(y, exog=X_its, order=orden, trend='n').fit(disp=False, maxiter=200)
            if mejor is None or modelo.aic < mejor[1]:
                mejor = (orden, modelo.aic, modelo)
        except Exception:
            continue
    orden, aic, modelo = mejor
    beta = modelo.params['clase']
    se = modelo.bse['clase']
    pct = (np.exp(beta) - 1) * 100 if c != 'PC1' else None
    resultados_its[c] = {'orden_ARMA': f'({orden[0]},{orden[2]})', 'beta': round(float(beta), 4),
                         'se': round(float(se), 4),
                         'pct': round(pct, 2) if pct is not None else None,
                         'p': round(float(modelo.pvalues['clase']), 6), 'aic': round(float(aic), 1)}
    etiqueta_pct = f"{pct:+.1f}%" if pct is not None else f"beta {beta:+.3f}"
    print(f"  {c}: ARMA{resultados_its[c]['orden_ARMA']} | {etiqueta_pct} | p = {resultados_its[c]['p']}")

ITS: SCHOOL-TERM INDICATOR (trend + Fourier + weekday + ARMA errors selected by AIC)


  CO: ARMA(2,1) | +5.7% | p = 0.186853


  NO2: ARMA(1,0) | +6.2% | p = 0.261589


  O3: ARMA(2,1) | -4.7% | p = 0.373456


  PM10: ARMA(1,0) | -2.5% | p = 0.72184


  PM2.5: ARMA(1,0) | -4.4% | p = 0.513823


  PC1: ARMA(1,0) | beta +0.107 | p = 0.529484


## 4. Cutoff-specific jumps: segmented regression over ±60 days with HAC errors

In [4]:
saltos_por_corte = {}
for corte in cortes_clase:
    t0 = pd.Timestamp(corte)
    ventana = serie_diaria[(serie_diaria['date'] >= t0 - pd.Timedelta(days=60)) &
                           (serie_diaria['date'] <= t0 + pd.Timedelta(days=60))].copy()
    ventana['rel'] = (ventana['date'] - t0).dt.days
    ventana['post'] = (ventana['rel'] >= 0).astype(float)
    X_salto = sm.add_constant(pd.concat([
        ventana[['post', 'rel']].astype(float),
        (ventana['rel'] * ventana['post']).rename('rel_post'),
        pd.get_dummies(ventana['dow'], prefix='dow', drop_first=True).astype(float)], axis=1))
    saltos = {}
    for c in contaminantes + ['PC1']:
        y = ventana[f'l_{c}'] if c != 'PC1' else ventana['PC1']
        modelo = sm.OLS(y, X_salto).fit(cov_type='HAC', cov_kwds={'maxlags': 14})
        beta = modelo.params['post']
        saltos[c] = {'salto': round(float(beta), 3), 'p': round(float(modelo.pvalues['post']), 4),
                     'pct': round((np.exp(beta) - 1) * 100, 1) if c != 'PC1' else None}
    saltos_por_corte[corte] = saltos

print("=" * 70)
print('CUTOFF-SPECIFIC JUMPS (±60 days, HAC): is the term-start jump consistent?')
print("=" * 70)
tabla_saltos = pd.DataFrame({corte: {c: f"{v[c]['pct']:+.0f}%{'*' if v[c]['p'] < 0.05 else ''}"
                                     for c in contaminantes}
                             for corte, v in saltos_por_corte.items()})
print(tabla_saltos)
print('\n(*) p < 0.05. A consistent jump would require the same sign across term starts.')

CUTOFF-SPECIFIC JUMPS (±60 days, HAC): is the term-start jump consistent?
      2023-01-09 2023-08-28 2024-01-08 2024-08-26 2025-01-09 2025-09-01
CO         +11%*        -3%        +4%      -11%*      +44%*      +16%*
NO2        +34%*       +12%        +3%       +12%        +9%        +9%
O3          +14%        -3%        +2%       +25%       -23%      +26%*
PM10         -1%       -20%        -4%       -19%       -13%       -17%
PM2.5      -34%*       -22%       -25%       -14%        -2%       -18%

(*) p < 0.05. A consistent jump would require the same sign across term starts.


## 5. RDiT: time discontinuities with a donut and reference cutoffs

### RDiT equation

Around each school-return date $t_0$, define relative day $r=t-t_0$ and exclude the transition donut $|r|\le2$:

$$ \log y_r=\alpha+\tau\mathbb{1}[r\ge0]+\phi_1r+\phi_2r\mathbb{1}[r\ge0]+\delta_d+\text{cutoff fixed effects}+u_r. $$

$\tau$ is the jump at the cutoff, allowing different slopes before ($\phi_1$) and after ($\phi_1+\phi_2$). Windows of ±28/42/56 days illustrate the tradeoff: narrower windows reduce exposure to broad seasonal changes but contain fewer observations; wider windows reverse that tradeoff. HAC (Newey–West) errors account for daily serial dependence.

### RDiT assumption

Local identification assumes that nothing else changes abruptly at the school-return date; smooth weather and seasonal changes are represented by local trends. The ±2-day donut removes immediate transition dates. Vacation-midpoint reference cutoffs examine this assumption: jumps there would caution against attributing a return-date jump specifically to school activity. This assumption is fragile here because January school returns coincide with broader work resumption. RDiT is therefore exploratory and does not define the paper's conclusions.

In [5]:
def rdit_agrupado(serie, cortes, ancho, donut=2):
    '\n    Pooled RDiT: windows of ±ancho days, excluding ±donut days around each cutoff,\n    with local trends on each side and HAC errors.\n    '
    ventanas = []
    for corte in cortes:
        t0 = pd.Timestamp(corte)
        w = serie[(serie['date'] >= t0 - pd.Timedelta(days=ancho)) &
                  (serie['date'] <= t0 + pd.Timedelta(days=ancho))].copy()
        w['rel'] = (w['date'] - t0).dt.days
        w = w[abs(w['rel']) > donut]
        w['post'] = (w['rel'] >= 0).astype(float)
        w['corte'] = corte
        ventanas.append(w)
    W = pd.concat(ventanas, ignore_index=True)
    X = pd.concat([W[['post', 'rel']].astype(float), (W['rel'] * W['post']).rename('rel_post'),
                   pd.get_dummies(W['dow'], prefix='dow', drop_first=True).astype(float),
                   pd.get_dummies(W['corte'], prefix='c', drop_first=True).astype(float)], axis=1)
    X = sm.add_constant(X)
    salida = {}
    for c in contaminantes:
        modelo = sm.OLS(W[f'l_{c}'], X).fit(cov_type='HAC', cov_kwds={'maxlags': 7})
        beta = modelo.params['post']
        se = modelo.bse['post']
        salida[c] = {'pct': round((np.exp(beta) - 1) * 100, 1),
                     'ci_lo_pct': round((np.exp(beta - 1.96 * se) - 1) * 100, 1),
                     'ci_hi_pct': round((np.exp(beta + 1.96 * se) - 1) * 100, 1),
                     'p': round(float(modelo.pvalues['post']), 4)}
    salida['n'] = int(len(W))
    return salida

resultados_rdit = {'clases': {}, 'placebo': {}}
for ancho in (28, 42, 56):
    resultados_rdit['clases'][f'±{ancho}d'] = rdit_agrupado(serie_diaria, cortes_clase, ancho)
for ancho in (28, 42):
    resultados_rdit['placebo'][f'±{ancho}d'] = rdit_agrupado(serie_diaria, cortes_placebo, ancho)

print("=" * 70)
print(f"POOLED RDiT AT {len(cortes_clase)} TERM STARTS (and reference cutoffs at vacation midpoints)")
print("=" * 70)
for ancho, r in resultados_rdit['clases'].items():
    print(f"\nWindow {ancho} (n = {r['n']}):")
    for c in contaminantes:
        e = r[c]
        marca = ' *' if e['p'] < 0.05 else ''
        print(f"  {c}: {e['pct']:+.1f}% [{e['ci_lo_pct']}, {e['ci_hi_pct']}] p = {e['p']}{marca}")
for ancho, r in resultados_rdit['placebo'].items():
    significativos = sum(1 for c in contaminantes if r[c]['p'] < 0.05)
    print(f"\nReference cutoffs {ancho}: {significativos} of 5 pollutants significant (n = {r['n']})")

POOLED RDiT AT 6 TERM STARTS (and reference cutoffs at vacation midpoints)

Window ±28d (n = 292):
  CO: +9.2% [-0.0, 19.4] p = 0.0504
  NO2: +17.4% [-2.6, 41.4] p = 0.0916
  O3: +9.5% [-7.6, 29.9] p = 0.2936
  PM10: +3.4% [-22.7, 38.3] p = 0.8224
  PM2.5: -4.2% [-24.9, 22.2] p = 0.7315

Window ±42d (n = 446):
  CO: +7.9% [-0.3, 16.8] p = 0.0608
  NO2: +13.2% [-2.1, 30.8] p = 0.0944
  O3: +6.6% [-8.5, 24.1] p = 0.411
  PM10: -9.7% [-28.3, 13.7] p = 0.3857
  PM2.5: -7.9% [-23.9, 11.3] p = 0.3934

Window ±56d (n = 600):
  CO: +7.8% [-1.8, 18.4] p = 0.1133
  NO2: +9.9% [-4.6, 26.5] p = 0.1901
  O3: +5.9% [-7.3, 21.0] p = 0.3995
  PM10: -15.8% [-30.7, 2.3] p = 0.0834
  PM2.5: -18.4% [-31.1, -3.4] p = 0.0184 *

Reference cutoffs ±28d: 1 of 5 pollutants significant (n = 312)

Reference cutoffs ±42d: 1 of 5 pollutants significant (n = 480)


In [6]:
fig, ax = plt.subplots(figsize=(13, 6.5))

anchos = [28, 42, 56]
for c in contaminantes:
    valores = [resultados_rdit['clases'][f'±{a}d'][c]['pct'] for a in anchos]
    ax.plot(anchos, valores, marker='o', linewidth=2, color=colores_contaminantes[c], label=c)

# CO confidence band
ci_lo = [resultados_rdit['clases'][f'±{a}d']['CO']['ci_lo_pct'] for a in anchos]
ci_hi = [resultados_rdit['clases'][f'±{a}d']['CO']['ci_hi_pct'] for a in anchos]
ax.fill_between(anchos, ci_lo, ci_hi, color='#3498db', alpha=0.15, label='CO 95% CI (transformed beta)')

ax.axhline(0, color='black', linewidth=1)
ax.set_xticks(anchos)
ax.set_xticklabels(['±28 days', '±42 days', '±56 days'])
ax.set_xlabel('Window width around term start', fontsize=12, fontweight='bold')
ax.set_ylabel('Estimated term-start jump (%)', fontsize=12, fontweight='bold')
ax.set_title('RDiT at term starts: sensitivity to window width\n(pooled cutoffs; vacation-midpoint reference results appear in the table above)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, ncol=2)
plt.tight_layout()
plt.savefig(graficos_dir / '03_rdit_sensibilidad_ventana.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {graficos_dir / '03_rdit_sensibilidad_ventana.png'}")

Plot saved to: ../data_processed/graficos_metodos/03_rdit_sensibilidad_ventana.png


## 6. GEE with within-unit AR(1) correlation: supplementary analysis

### GEE equation

Use the same mean specification as ITS, estimated by generalized estimating equations with within-unit working correlation $\operatorname{corr}(u_t,u_{t-s})=\rho^s$. The sandwich variance clusters by monitoring unit. There are only six units, which limits the reliability of standard cluster asymptotics.

In [7]:
X_gee = pd.concat([panel[['clase']].astype(float), panel[columnas_fourier],
                   (panel['t'] / 365.25).rename('tendencia'),
                   pd.get_dummies(panel['dow'], prefix='dow', drop_first=True).astype(float)], axis=1)
X_gee = sm.add_constant(X_gee)

resultados_gee = {}
print("=" * 70)
print('GEE AR(1) BY UNIT (only six clusters: interpret cautiously; secondary method)')
print("=" * 70)
for c in contaminantes:
    try:
        modelo = GEE(panel[f'l_{c}'], X_gee, groups=panel['estacion'], time=panel['t'].values,
                     cov_struct=Autoregressive(grid=True), family=sm.families.Gaussian()).fit(maxiter=100)
        beta = modelo.params['clase']
        resultados_gee[c] = {'pct': round((np.exp(beta) - 1) * 100, 2),
                             'p': round(float(modelo.pvalues['clase']), 6),
                             'ar1': round(float(getattr(modelo.cov_struct, 'dep_params', np.nan)), 3)}
        print(f"  {c}: {resultados_gee[c]['pct']:+.1f}% | p = {resultados_gee[c]['p']} | AR1 = {resultados_gee[c]['ar1']}")
    except Exception as e:
        resultados_gee[c] = {'error': str(e)[:120]}
print('=> With six clusters, sandwich standard errors may be misleadingly small')
print('   (Cameron & Miller 2015); these estimates are documented but not used for main inference.')

GEE AR(1) BY UNIT (only six clusters: interpret cautiously; secondary method)
  CO: +3.6% | p = 0.0 | AR1 = 0.783
  NO2: +6.1% | p = 0.0 | AR1 = 0.548
  O3: -4.9% | p = 0.01321 | AR1 = 0.394
  PM10: -2.5% | p = 0.013755 | AR1 = 0.465


  PM2.5: -4.4% | p = 0.011618 | AR1 = 0.435
=> With six clusters, sandwich standard errors may be misleadingly small
   (Cameron & Miller 2015); these estimates are documented but not used for main inference.


In [8]:
with open(resultados_dir / 'its_rdit_gee.json', 'w') as f:
    json.dump({'ITS_estado': resultados_its, 'saltos_por_corte': saltos_por_corte,
               'RDiT': resultados_rdit, 'GEE': resultados_gee,
               'cortes_clase': cortes_clase, 'cortes_placebo': cortes_placebo},
              f, indent=2, ensure_ascii=False)
print(f"Saved to: {resultados_dir / 'its_rdit_gee.json'}")

print('\nCOMPLETED')

Saved to: ../data_processed/resultados_metodos/its_rdit_gee.json

COMPLETED
